# model

> The learner model: how deep you know each concept, when to review it, and what to ask next.

This module holds the learner model as plain functions with no model calls and no storage.
It finds what an answer means, updates the state of a concept, finds the next depth and picks the next concept.
The tutor calls these functions and the store keeps their results.

In [ ]:
#| default_exp model

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| export
import time
from fastcore.utils import *

## Depth

Each concept has a depth from 0 to 5. The tutor asks each question at one depth.

| depth | name | a question at this depth asks you to |
|---|---|---|
| 0 | unseen | (no question) |
| 1 | recognise | pick the correct statement from options |
| 2 | recall | state the idea or the rule |
| 3 | explain | say why it is true, in your own words |
| 4 | apply | use it to solve a problem |
| 5 | transfer | use it in a new case or link it to another subject |

The floor is the highest depth that you pass reliably. The ceiling is the lowest depth that you failed (6 until a fail).
The next question goes between them, so each answer moves one edge of this range.

These constants name the depths, give the question kind at each depth, and set the goal and the high-confidence line.

In [ ]:
#| export
DEPTHS = ('unseen', 'recognise', 'recall', 'explain', 'apply', 'transfer')
KIND_AT = {1: 'choice', 2: 'recall', 3: 'explain', 4: 'apply', 5: 'transfer'}
GOAL = 4      # the default depth that counts as learned
HIGH = 60     # confidence at or above this is "high"
DAY = 86400.

## Outcome

Correctness and confidence together tell what an answer means.

| | high confidence | low confidence |
|---|---|---|
| correct | `solid` | `fragile` |
| wrong | `misconception` | `gap` |

"I don't know" is a `gap`. A correct answer that used two or more hints is `fragile`.

In [ ]:
#| export
def outcome(correct:bool, confidence:int=50, hints:int=0, dont_know:bool=False) -> str:
    "One of `solid`, `fragile`, `misconception` or `gap`."
    if dont_know: return 'gap'
    if correct: return 'solid' if confidence >= HIGH and hints < 2 else 'fragile'
    return 'misconception' if confidence >= HIGH else 'gap'

A correct answer is `solid` with high confidence. Two hints or low confidence make it `fragile`.

In [ ]:
test_eq(outcome(True, 80), 'solid')
test_eq(outcome(True, 80, hints=2), 'fragile')
test_eq(outcome(True, 30), 'fragile')

A wrong answer is a `misconception` with high confidence, else a `gap`. "I don't know" is always a `gap`.

In [ ]:
test_eq(outcome(False, 90), 'misconception')
test_eq(outcome(False, 20), 'gap')
test_eq(outcome(True, 90, dont_know=True), 'gap')

## Memory

Recall falls with time as in FSRS: $R(t) = (1 + \frac{19}{81}\frac{t}{S})^{-1/2}$. The time $t$ and the stability $S$ are in days.
At $t = S$, recall is 90%. Thus the due date is the last review plus $S$ days.

`recall` gives the recall now. `grow` gives the new stability after an outcome.

In [ ]:
#| export
def recall(c, now:float=None) -> float:
    "The chance, from 0 to 1, that you still recall concept `c` now."
    if not c.get('stability') or c.get('last') is None: return 0.
    t = (ifnone(now, time.time()) - c['last']) / DAY
    return (1 + 19/81 * max(t, 0) / c['stability']) ** -.5

def grow(s:float, r:float, kind:str) -> float:
    "The new stability after an outcome, from the old stability `s` and the recall `r` at the time of the answer."
    if kind == 'solid': return 1. if not s else s * (1 + 3 * (1 - r) + .2)
    if kind == 'fragile': return .5 if not s else s * (1 + (1 - r) + .05)
    if kind == 'misconception': return .2
    return max(.2, s * .3)

Recall is 90% after $S$ days, and 0 for a concept with no review.

In [ ]:
test_close(recall(dict(stability=2., last=0.), now=2*DAY), .9, eps=1e-6)
test_eq(recall(dict(stability=0, last=None)), 0.)

A pass grows stability. A harder recall grows it more. A gap shrinks it.

In [ ]:
test_eq(grow(0, 0, 'solid'), 1.)
assert grow(4, .7, 'solid') > grow(4, .99, 'solid') > 4   # a review that was harder to recall teaches more
assert grow(4, .9, 'gap') < 4

## Update

`update` returns the fields to change after one answer at depth `d`.

| outcome | change |
|---|---|
| `solid` | The floor rises to `d`. The streak grows. |
| `fragile` | The floor stays. The concept is `shaky` at `d`, so the next question is at `d` again. |
| `misconception` | The ceiling falls to `d`. The floor falls below `d`. The misconception is kept. |
| `gap` | The ceiling falls to `d`. The floor falls below `d`. |

In [ ]:
#| export
def update(c, depth:int, correct:bool, confidence:int=50, hints:int=0, dont_know:bool=False,
           misconception:str='', now:float=None) -> AttrDict:
    "The state changes for concept `c` after one answer at `depth`, with the outcome in `kind`."
    now, k = ifnone(now, time.time()), outcome(correct, confidence, hints, dont_know)
    fl, ce, st, mi = c.get('floor', 0), c.get('ceiling', 6), c.get('streak', 0), c.get('misses', 0)
    shaky, mis = c.get('shaky'), list(c.get('misconceptions') or [])
    if k == 'solid':
        fl, st, mi, shaky = max(fl, depth), st + 1, 0, None
        if ce <= fl: ce = fl + 1
    elif k == 'fragile': st, mi, shaky = 0, 0, depth if depth > fl else None
    else:
        ce, fl, st, mi, shaky = min(ce, depth), min(fl, depth - 1), 0, mi + 1, None
        if k == 'misconception' and misconception and misconception not in mis: mis = [*mis, misconception][-5:]
    s = grow(c.get('stability', 0), recall(c, now), k)
    return AttrDict(kind=k, floor=max(fl, 0), ceiling=min(ce, 6), streak=st, misses=mi, shaky=shaky, misconceptions=mis,
                    stability=s, last=now, due=now + s * DAY, seen=c.get('seen', 0) + 1)

A `solid` pass at depth 3 on a new concept raises the floor to 3. The review is due in one day.

In [ ]:
c = AttrDict(floor=0, ceiling=6, stability=0., last=None, streak=0, misses=0, seen=0)
u = update(c, 3, True, 80, now=0.)
test_eq((u.kind, u.floor, u.ceiling, u.streak, u.due), ('solid', 3, 6, 1, DAY))
c.update(u)

A confident wrong answer at depth 4 lowers the ceiling to 4 and keeps the misconception.

In [ ]:
u = update(c, 4, False, 85, misconception='thinks RRF needs normalised scores', now=DAY)
test_eq((u.kind, u.floor, u.ceiling, u.misses), ('misconception', 3, 4, 1))
test_eq(u.misconceptions, ['thinks RRF needs normalised scores'])
c.update(u)

A miss below the floor lowers the floor. A low-confidence pass makes the concept `shaky` at that depth.

In [ ]:
u = update(c, 2, False, 10, now=DAY)    # a miss below the floor: the floor falls
test_eq((u.kind, u.floor, u.ceiling), ('gap', 1, 2))
test_eq(update(c, 4, True, 30).shaky, 4)

## The next depth

Ask in the middle of the range between floor and ceiling, so one answer cuts the range in half.
Ask a `shaky` concept at the same depth again.
When the ceiling is the floor plus 1, ask at the ceiling, because the work is at that edge.

In [ ]:
#| export
def next_depth(c) -> int:
    "The depth for the next question on concept `c`."
    if c.get('shaky'): return c['shaky']
    fl, ce = c.get('floor', 0), min(c.get('ceiling', 6), 6)
    if fl >= 5: return 5
    return min(5, fl + max(1, (ce - fl) // 2))

A new concept starts at depth 3 (explain). An open range gives its middle.

In [ ]:
test_eq(next_depth(dict(floor=0, ceiling=6)), 3)     # a new concept: start at explain, not at the bottom
test_eq(next_depth(dict(floor=3, ceiling=6)), 4)
test_eq(next_depth(dict(floor=0, ceiling=3)), 1)
test_eq(next_depth(dict(floor=1, ceiling=3)), 2)

A closed range gives the ceiling. Depth 5 is the top. A `shaky` depth comes first.

In [ ]:
test_eq(next_depth(dict(floor=2, ceiling=3)), 3)
test_eq(next_depth(dict(floor=5, ceiling=6)), 5)
test_eq(next_depth(dict(floor=1, ceiling=6, shaky=4)), 4)

## What to ask next

`pick` chooses the concept. Code makes the choice, not the model, so the app can show the reason.

1. A `focus` concept, when you chose one and did not learn it yet.
2. A review that is due, with the lowest recall first.
3. The frontier: concepts below the goal with all prereqs at depth 3 or more. Concepts that you started come first.

After two misses in a row on a concept, `pick` goes back to its weakest prereq that is below the goal.

`learned` and `ready` test one concept. `_back` finds the prereq to go back to.

In [ ]:
#| export
def learned(c, goal:int=GOAL) -> bool: return c.get('floor', 0) >= goal

def ready(c, by:dict) -> bool:
    "Whether each prereq of `c` that is a known concept is at depth 3 or more."
    return all(by[p].get('floor', 0) >= 3 for p in c.get('prereqs', ()) if p in by)

def _back(c, by, goal):
    "The weakest prereq of `c` that is not learned, after two misses; else `c`."
    if c.get('misses', 0) < 2: return c, ''
    ps = [by[p] for p in c.get('prereqs', ()) if p in by and not learned(by[p], goal)]
    if not ps: return c, ''
    p = min(ps, key=lambda x: x.get('floor', 0))
    return p, f"two misses on {c['name']}; step back to {p['name']}"

`pick` returns the concept, the depth and the reason.

In [ ]:
#| export
def pick(concepts, focus:str=None, subject:str=None, goal:int=GOAL, now:float=None, skip=()):
    "The next `(concept, depth, why)`, or None when no review is due and the frontier is empty."
    now = ifnone(now, time.time())
    by = {c['slug']: c for c in concepts}
    pool = [c for c in concepts if c['slug'] not in skip and (not subject or c.get('subject') == subject)]
    def out(c, why):
        c, back = _back(c, by, goal)
        return c, next_depth(c) if not learned(c, goal) else c.get('floor', 0), back or why
    if focus and (f := by.get(focus)) and (not learned(f, goal) or (f.get('due') or 0) <= now):
        return out(f, f"you chose {f['name']}")
    due = [c for c in pool if c.get('seen') and c.get('due') and c['due'] <= now]
    if due:
        c = min(due, key=lambda c: recall(c, now))
        return out(c, f"due for review: recall about {recall(c, now):.0%}")
    front = [c for c in pool if not learned(c, goal) and ready(c, by)]
    if not front: return None
    c = min(front, key=lambda c: (-bool(c.get('seen')), c.get('order', 1e9), -c.get('floor', 0)))
    return out(c, f"next on the frontier: {'you started it' if c.get('seen') else 'its prereqs are in place'}")

Three concepts: `b` needs `a`. The first pick is `a`, because `b` waits for its prereq.

In [ ]:
def C(slug, **kw): return AttrDict(slug=slug, name=slug, **{**dict(floor=0, ceiling=6, seen=0, misses=0, prereqs=[]), **kw})
cs = [C('a', order=0), C('b', prereqs=['a'], order=1), C('c', order=2)]
test_eq(pick(cs)[0].slug, 'a')                          # b waits for a

A started concept below the goal comes first. When `a` is learned, `b` is ready.

In [ ]:
cs[0].update(floor=3, seen=1, due=10*DAY, stability=10., last=0.)
test_eq(pick(cs, now=DAY)[0].slug, 'a')                 # a was started and is below the goal
cs[0].update(floor=4)
test_eq(pick(cs, now=DAY)[0].slug, 'b')                 # a is learned; b is ready

A due review comes before the frontier, at the floor of the concept.

In [ ]:
r = pick(cs, now=11*DAY)
test_eq((r[0].slug, r[1]), ('a', 4))                    # a is due: a review at its floor
assert 'due for review' in r[2]

Two misses go back to the weakest prereq. A learned concept that is not due gives no question.

In [ ]:
cs[1].update(misses=2, seen=2, floor=1, ceiling=2); cs[0].update(floor=3, due=None)
test_eq(pick(cs, focus='b', now=DAY)[0].slug, 'a')      # two misses on b: step back to a
test_eq(pick([C('x', floor=4, seen=1, due=1e12)]), None)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()